Esegui lo split del file delle caption in tre file separati

In [12]:
import pandas as pd


CAPTIONS_DIR = "../data/captions"
CAPTIONS_FILE = f"{CAPTIONS_DIR}/captions.csv"
FLICKR8K_TEXT_DIR = "../Flickr8k/Flickr8k_text"


def split_captions_file(
        captions_file,

        train_images_file,
        train_captions_file,
        
        val_images_file,
        val_captions_file,
        
        test_images_file, 
        test_captions_file
    ):
    
    # Training captions
    split(
        captions_file,
        train_images_file,
        train_captions_file  
    )

    # Validation captions
    split(
        captions_file,
        val_images_file,    
        val_captions_file
    )

    # Test captions
    split(
        captions_file,
        test_images_file, 
        test_captions_file
    )


def split(captions_file, images_file, out_file):

    captions_df = pd.read_csv(captions_file)

    df = pd.read_csv(
        images_file,
        header=None,
        names=["image"]
    )

    df["image"] = df["image"].str.replace(
        ".jpg", 
        "", 
        regex=False
    )

    new_df = df.merge(
        captions_df,
        on="image",
        how="inner"
    )

    new_df.to_csv(out_file, index=False)


split_captions_file(
    captions_file=CAPTIONS_FILE,
    
    train_images_file=f"{FLICKR8K_TEXT_DIR}/Flickr_8k.trainImages.txt",
    train_captions_file=f"{CAPTIONS_DIR}/train_captions.csv",
    
    val_images_file=f"{FLICKR8K_TEXT_DIR}/Flickr_8k.devImages.txt",
    val_captions_file=f"{CAPTIONS_DIR}/val_captions.csv",

    test_images_file=f"{FLICKR8K_TEXT_DIR}/Flickr_8k.testImages.txt",
    test_captions_file=f"{CAPTIONS_DIR}/test_captions.csv"
)

Crea un dizionario a partire dal file delle caption di training

In [10]:
import pandas as pd


def load_captions(path):
    df = pd.read_csv(path)
    return (
        df.groupby("image")["caption"]
          .apply(list)
          .to_dict()
    )


train_captions = load_captions("../data/captions/train_captions.csv")

Calcola la lunghezza massima delle caption di training

In [11]:
import numpy as np


def compute_max_len(captions: dict[str, list[str]], percentile: int = 99):
    lengths = [len(caption.split()) for image_captions in captions.values() for caption in image_captions]
    return int(np.percentile(lengths, percentile))


max_len = compute_max_len(train_captions, percentile=97)
print(f"Max Len: {max_len}")

Max Len: 15


Trocare le caption a max_len

In [13]:
def truncate_captions(captions: dict[str, list[str]], max_len: int):
    truncated_captions = {}
    count_truncated = 0
    for image, image_captions in captions.items():
        truncated_image_captions = []
        for caption in image_captions:
            words = caption.split()
            if len(words) > max_len:
                truncated_caption = " ".join(words[:max_len])
                truncated_image_captions.append(truncated_caption)
                count_truncated += 1
            else:
                truncated_image_captions.append(caption)
        truncated_captions[image] = truncated_image_captions
    return truncated_captions, count_truncated


truncated_train_captions, count_truncated = truncate_captions(train_captions, max_len)
print(f"Truncated {count_truncated} captions to max length of {max_len} words.")

Truncated 631 captions to max length of 15 words.


Aggiungi i token [START] ed [END]

In [15]:
# add [START] and [END] tokens to the captions

def add_start_end_tokens(captions: dict[str, list[str]]):
    captions_with_tokens = {}
    for image, image_captions in captions.items():
        image_captions_with_tokens = []
        for caption in image_captions:
            caption_with_tokens = f"[START] {caption} [END]"
            image_captions_with_tokens.append(caption_with_tokens)
        captions_with_tokens[image] = image_captions_with_tokens
    return captions_with_tokens


truncated_train_captions_with_tokens = add_start_end_tokens(truncated_train_captions)

print(f"First 5 truncated captions with tokens:")
for i, (image, image_captions) in enumerate(truncated_train_captions_with_tokens.items()):
    print(f"  {image_captions[0]}")
    if i >= 4:
        break

First 5 truncated captions with tokens:
  [START] child in pink dress climbing up set of stairs in entry way [END]
  [START] black dog and spotted dog fighting [END]
  [START] little girl covered in paint sits in front of painted rainbow with hands in bowl [END]
  [START] man lays on bench while dog sits by him [END]
  [START] man in orange hat starring at something [END]


Crea un vettorizzatore e adattalo alle caption di training

In [21]:
from keras.layers import TextVectorization


def create_vetorizer(output_sequence_length: int, captions: dict[str, list[str]]) -> TextVectorization:
    vec = TextVectorization(
        max_tokens=None,
        output_sequence_length=output_sequence_length,
        split="whitespace",
        standardize=None, # type: ignore
        output_mode="int",
        name="text_vectorization"
    )

    caption_list = [caption for image_captions in truncated_train_captions_with_tokens.values() for caption in image_captions]

    vec.adapt(caption_list)

    return vec


# Non abbiamo rimosso le parole con una freq bassa
# per mantenere il senso delle frasi
output_sequence_length = max_len + 2
vectorizer = create_vetorizer(output_sequence_length, truncated_train_captions_with_tokens)
print(f"Vocabulary size: {len(vectorizer.get_vocabulary())}")
print(f"Vocabulary: {vectorizer.get_vocabulary()}")

Vocabulary size: 7578
Vocabulary: ['', '[UNK]', '[START]', '[END]', 'in', 'on', 'and', 'dog', 'with', 'man', 'of', 'two', 'white', 'black', 'boy', 'woman', 'girl', 'to', 'wearing', 'at', 'people', 'water', 'brown', 'young', 'red', 'blue', 'dogs', 'running', 'through', 'playing', 'while', 'down', 'shirt', 'ball', 'standing', 'little', 'grass', 'snow', 'child', 'person', 'jumping', 'over', 'three', 'sitting', 'field', 'front', 'holding', 'small', 'yellow', 'green', 'group', 'up', 'large', 'by', 'walking', 'one', 'men', 'children', 'air', 'into', 'near', 'beach', 'mouth', 'jumps', 'runs', 'another', 'street', 'for', 'riding', 'from', 'stands', 'bike', 'girls', 'as', 'outside', 'play', 'rock', 'looking', 'other', 'orange', 'out', 'pink', 'player', 'next', 'off', 'camera', 'pool', 'jacket', 'hat', 'boys', 'around', 'women', 'behind', 'toy', 'soccer', 'some', 'sits', 'wall', 'has', 'dressed', 'dirt', 'background', 'walks', 'plays', 'mountain', 'stand', 'along', 'park', 'football', 'climbing'

Vettorizza le caption, teacher forcing sul risultato

In [26]:
from tensorflow import Tensor

def vectorize_captions(vec: TextVectorization, captions: dict[str, list[str]]) -> Tensor:
    # L'ordine delle caption è preservato anche nella lista
    all_captions = [caption for image_captions in captions.values() for caption in image_captions]
    return vec(all_captions)


def split_captions_for_teacher_forcing(captions: Tensor) -> tuple[Tensor, Tensor]:
    input_captions = captions[:, :-1]
    target_captions = captions[:, 1:]
    return input_captions, target_captions


captions_tensor = vectorize_captions(vectorizer, truncated_train_captions_with_tokens)

print(f"\nType of vec ret value: {type(captions_tensor)}")

captions_input, captions_target = split_captions_for_teacher_forcing(captions_tensor)

print(f"\nType of captions_input: {type(captions_input)}")

print(f"Captions tensor shape: {captions_tensor.shape}")
print(f"Input captions shape: {captions_input.shape}")
print(f"Target captions shape: {captions_target.shape}")


Type of vec ret value: <class 'tensorflow.python.framework.ops.EagerTensor'>

Type of captions_input: <class 'tensorflow.python.framework.ops.EagerTensor'>
Captions tensor shape: (30000, 17)
Input captions shape: (30000, 16)
Target captions shape: (30000, 16)


Crea un dict in cui le chiavi sono le immagini e i valori sono dict con chiavi "input" e "target". "input" è una tensore di shape (5, 16) e "target" anche. SOno rispettivamente le caption di input e quelle di target (vettorizzate) corrispondenti alla immagine chiave.

In [24]:
new = {}
for i, image in enumerate(truncated_train_captions_with_tokens):
    start = i * 5
    end = start + 5

    new[image] = {
        "input": captions_input[start:end],
        "target": captions_target[start:end],
    }

# Print first entry
print(f"First entry: {list(new.items())[0]}")

First entry: ('1000268201_693b08cb0e', {'input': <tf.Tensor: shape=(5, 16), dtype=int64, numpy=
array([[   2,   38,    4,   81,  163,  109,   51,  386,   10,  391,    4,
        6652,  642,    3,    0,    0],
       [   2,   16,  305,   59,  188,  115,    3,    0,    0,    0,    0,
           0,    0,    0,    0,    0],
       [   2,   35,   16,  109,   59,  188, 2117,    3,    0,    0,    0,
           0,    0,    0,    0,    0],
       [   2,   35,   16,  109,  391,   17, 2117,    3,    0,    0,    0,
           0,    0,    0,    0,    0],
       [   2,   35,   16,    4,   81,  163,  305,   59,  188, 3373,    3,
           0,    0,    0,    0,    0]])>, 'target': <tf.Tensor: shape=(5, 16), dtype=int64, numpy=
array([[  38,    4,   81,  163,  109,   51,  386,   10,  391,    4, 6652,
         642,    3,    0,    0,    0],
       [  16,  305,   59,  188,  115,    3,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0],
       [  35,   16,  109,   59,  188, 2117,    3,    

TF Record: tutte le info sono in new. Devi capire come serializzare...

In [ ]:
# Come creare un TF Record con il contenuto di new?

# ???
def create_tf_record(new: dict[str, dict[str, Tensor]], output_file: str):
    import tensorflow as tf

    with tf.io.TFRecordWriter(output_file) as writer:
        for image, data in new.items():
            input_captions = data["input"].numpy().tolist()
            target_captions = data["target"].numpy().tolist()

            feature = {
                "image": tf.train.Feature(bytes_list=tf.train.BytesList(value=[image.encode()])),
                "input_captions": tf.train.Feature(int64_list=tf.train.Int64List(value=[item for sublist in input_captions for item in sublist])),
                "target_captions": tf.train.Feature(int64_list=tf.train.Int64List(value=[item for sublist in target_captions for item in sublist])),
            }

            example = tf.train.Example(features=tf.train.Features(feature=feature))
            writer.write(example.SerializeToString())

    print(f"TFRecord created at: {output_file}")

Tutto il procedimento va ripeto anche per val. Anche per test?